In [1]:
import time
import cv2
import serial
import serial.tools.list_ports
from cvzone.HandTrackingModule import HandDetector

print("Puertos serie detectados:")
for p in serial.tools.list_ports.comports():
    print(f"  {p.device:8}  {p.description}")

Puertos serie detectados:
  COM5      Silicon Labs CP210x USB to UART Bridge (COM5)
  COM4      Serie estándar sobre el vínculo Bluetooth (COM4)
  COM3      Serie estándar sobre el vínculo Bluetooth (COM3)


In [2]:
PUERTO = "COM5"      # puerto del ESP32
BAUDIOS = 115200     # debe coincidir con Serial.begin() del sketch
CAMARA = 0           # 0 = cámara integrada; prueba 1 si usas una USB
FRAMES_ESTABLES = 5  # cuántos fotogramas seguidos debe mantenerse el gesto antes de cambiar el foco

In [3]:
try:
    esp = serial.Serial(PUERTO, BAUDIOS, timeout=0.1)
    time.sleep(2)  # el ESP32 se reinicia al abrir el puerto; esperamos a que arranque
    esp.reset_input_buffer()
    print(f"Conectado a {PUERTO}")
except serial.SerialException as e:
    esp = None
    print(f"No se pudo abrir {PUERTO}: {e}\nModo simulación: el foco no se controlará.")


def enviar(estado):
    """Envía 1 (encender) o 0 (apagar) al ESP32."""
    if esp is not None and esp.is_open:
        esp.write(b"1\n" if estado else b"0\n")
        respuesta = esp.readline().decode(errors="ignore").strip()
        if respuesta:
            print("ESP32:", respuesta)

Conectado a COM5


In [ ]:
cap = cv2.VideoCapture(CAMARA)
detector = HandDetector(maxHands=1, detectionCon=0.8)

foco_encendido = False
enviar(False)               # empezamos con el foco apagado
gesto_previo, contador = None, 0

try:
    while True:
        ok, img = cap.read()
        if not ok:
            print("No se pudo leer la cámara. Revisa el valor de CAMARA.")
            break

        hands, img = detector.findHands(img, draw=True)
        gesto = None
        dedos = None
        if hands:
            dedos = detector.fingersUp(hands[0]).count(1)
            if dedos == 5:
                gesto = "abierta"
            elif dedos == 0:
                gesto = "cerrada"

        # Solo cambiamos el foco si el gesto se mantiene varios fotogramas (evita parpadeos)
        if gesto is not None and gesto == gesto_previo:
            contador += 1
        else:
            contador = 1 if gesto else 0
        gesto_previo = gesto

        if contador >= FRAMES_ESTABLES:
            nuevo = gesto == "abierta"
            if nuevo != foco_encendido:
                foco_encendido = nuevo
                enviar(foco_encendido)
                print("Foco", "ENCENDIDO" if foco_encendido else "APAGADO")

        # Información en pantalla
        color = (0, 220, 255) if foco_encendido else (90, 90, 90)
        cv2.circle(img, (50, 50), 30, color, cv2.FILLED)
        cv2.putText(img, "FOCO ON" if foco_encendido else "FOCO OFF", (95, 62),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.1, color, 3)
        texto = f"Dedos: {dedos}" if dedos is not None else "Sin mano"
        cv2.putText(img, texto, (20, img.shape[0] - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        if esp is None:
            cv2.putText(img, "SIMULACION (sin ESP32)", (20, 120),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        cv2.imshow("Foco con la mano  (q para salir)", img)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()

ESP32: OK:0
ESP32: OK:1
Foco ENCENDIDO
ESP32: OK:0
Foco APAGADO
ESP32: OK:1
Foco ENCENDIDO
ESP32: OK:0
Foco APAGADO
ESP32: OK:1
Foco ENCENDIDO
ESP32: OK:0
Foco APAGADO
ESP32: OK:1
Foco ENCENDIDO
ESP32: OK:0
Foco APAGADO
ESP32: OK:1
Foco ENCENDIDO
ESP32: OK:0
Foco APAGADO
ESP32: OK:1
Foco ENCENDIDO
ESP32: OK:0
Foco APAGADO
ESP32: OK:1
Foco ENCENDIDO
ESP32: OK:0
Foco APAGADO
ESP32: OK:1
Foco ENCENDIDO
ESP32: OK:0
Foco APAGADO
ESP32: OK:1
Foco ENCENDIDO
ESP32: OK:0
Foco APAGADO
ESP32: OK:1
Foco ENCENDIDO
ESP32: OK:0
Foco APAGADO


In [ ]:
if esp is not None and esp.is_open:
    enviar(False)
    esp.close()
    print("Puerto cerrado.")